# Classificador de Despesas Bancarias — Zero-Shot Classification

Categorizacao automatica de transacoes bancarias usando modelo pre-treinado do Hugging Face.

**Tecnica:** Zero-Shot Classification — o modelo classifica textos em categorias que nunca viu no treinamento.

**Modelo:** `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` (multilingual, treinado em NLI, funciona bem em portugues)

## Celula 1 — Instalacao e Importacao

In [ ]:
# Instala as bibliotecas (no Colab geralmente ja esta disponivel)
!pip install transformers torch -q

from transformers import pipeline
import pandas as pd

## Celula 2 — Dados de Simulacao

Criando uma lista de transacoes bancarias simulando o extrato de um cliente.

In [ ]:
# Lista de descricoes de transacoes bancarias
transacoes = [
    "Compra no supermercado Pao de Acucar",
    "Pagamento Uber viagem centro",
    "Conta de luz Enel Sao Paulo",
    "Ingresso cinema Cinemark shopping",
    "Pedido iFood hamburgueria",
    "Posto Shell abastecimento gasolina",
    "Mensalidade Netflix streaming",
    "Compra Magazine Luiza eletronicos",
    "Farmacia Drogasil medicamentos",
    "Academia Smart Fit mensalidade"
]

# Categorias candidatas para classificacao
categorias_candidatas = [
    'Alimentacao',
    'Transporte',
    'Contas e Servicos',
    'Lazer',
    'Compras',
    'Saude'
]

print(f"Total de transacoes: {len(transacoes)}")
print(f"Categorias possiveis: {categorias_candidatas}")

## Celula 3 — Carregando o Classificador Zero-Shot

Usamos `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` em vez do `vidhur2k/mBERT-Portuguese-Mono` porque ele foi treinado em NLI (Natural Language Inference), que e exatamente o que a pipeline `zero-shot-classification` precisa para funcionar.

In [ ]:
# Criando o classificador zero-shot com modelo multilingual
classificador = pipeline(
    task='zero-shot-classification',
    model='MoritzLaurer/mDeBERTa-v3-base-mnli-xnli'
)

print("Modelo carregado com sucesso!")

## Celula 4 — Testando com um Unico Exemplo

Antes de rodar em todas as transacoes, testamos com uma unica para entender o formato de saida.

In [ ]:
# Testando com a primeira transacao
exemplo = transacoes[0]  # "Compra no supermercado Pao de Acucar"

resultado = classificador(exemplo, categorias_candidatas)

print(f"Transacao analisada: {exemplo}\n")
print("Resultados (ordenados por probabilidade):")
for label, score in zip(resultado['labels'], resultado['scores']):
    print(f"  {label}: {score:.4f} ({score*100:.2f}%)")

## Celula 5 — Classificando Todas as Transacoes

Aplicando o classificador em loop para cada transacao e extraindo a categoria com maior score.

In [ ]:
# Listas para armazenar resultados
categorias_previstas = []
scores_confianca = []
segundas_categorias = []
segundas_scores = []

# Classificando cada transacao
for transacao in transacoes:
    resultado = classificador(transacao, categorias_candidatas)

    # Categoria com maior score (primeiro item, ja vem ordenado)
    categoria_top = resultado['labels'][0]
    score_top = resultado['scores'][0]

    # Segunda categoria (para analise de ambiguidade)
    categoria_2 = resultado['labels'][1]
    score_2 = resultado['scores'][1]

    categorias_previstas.append(categoria_top)
    scores_confianca.append(round(score_top, 4))
    segundas_categorias.append(categoria_2)
    segundas_scores.append(round(score_2, 4))

    print(f"  {transacao:<45} -> {categoria_top} ({score_top*100:.1f}%)")

## Celula 6 — DataFrame com Resultados

Organizando tudo em um DataFrame com deteccao de transacoes ambiguas (diferenca < 20% entre 1a e 2a categoria).

In [ ]:
# Criando o DataFrame final
df_resultados = pd.DataFrame({
    'Transacao': transacoes,
    'Categoria Prevista': categorias_previstas,
    'Confianca (%)': [round(s * 100, 2) for s in scores_confianca],
    '2a Opcao': segundas_categorias,
    'Score 2a (%)': [round(s * 100, 2) for s in segundas_scores]
})

# Indicador de ambiguidade
df_resultados['Diferenca (%)'] = (
    df_resultados['Confianca (%)'] - df_resultados['Score 2a (%)']
)
df_resultados['Ambiguo?'] = df_resultados['Diferenca (%)'].apply(
    lambda x: 'Sim' if x < 20 else 'Nao'
)

df_resultados

## Celula 7 — Analise dos Resultados

In [ ]:
# Distribuicao de gastos por categoria
print("=== DISTRIBUICAO DE GASTOS POR CATEGORIA ===\n")
resumo = df_resultados['Categoria Prevista'].value_counts()
print(resumo)

print("\n=== TRANSACOES AMBIGUAS (diferenca < 20%) ===\n")
ambiguas = df_resultados[df_resultados['Ambiguo?'] == 'Sim']
if len(ambiguas) > 0:
    print(ambiguas[['Transacao', 'Categoria Prevista', '2a Opcao', 'Diferenca (%)']].to_string(index=False))
else:
    print("Nenhuma transacao ambigua identificada!")

print(f"\nConfianca media: {df_resultados['Confianca (%)'].mean():.2f}%")

## Reflexoes Finais

**1. Escolha do Modelo:** Modelos multilingues como `mDeBERTa-v3-base-mnli-xnli` ou `xlm-roberta-large-xnli` performam melhor em portugues para zero-shot do que modelos puramente mBERT, pois foram fine-tunados em tarefas de NLI — que e a base teorica do zero-shot classification.

**2. Definicao das Categorias:** Se tivessemos categorias muito parecidas como "Restaurante" e "Alimentacao", o modelo distribuiria os scores entre as duas, gerando ambiguidade. A recomendacao e usar categorias mutuamente exclusivas e bem definidas.

**3. Resultados Ambiguos:** Transacoes como "Padaria do bairro" poderiam ficar entre "Alimentacao" e "Compras". O monitoramento da diferenca entre o 1o e 2o score (como implementado acima) ajuda a sinalizar casos que precisariam de revisao humana ou regras adicionais.

**Proximos passos:**
- Integrar com API bancaria real (Open Finance)
- Aplicar fine-tuning com dados rotulados do proprio banco
- Criar camada de regras (regex) para padroes especificos como "PIX recebido" ou "TED enviada"